# Answer Reviewer — Fine-tuning Mistral-7B with QLoRA

Fine-tunes Mistral-7B-Instruct to act as an **automatic short answer grader**.

Given a question, a reference answer, and a student's answer — the model outputs:
- A grade label: `correct`, `partially_correct`, or `incorrect`
- A brief explanation of why

**Dataset:** [Meyerger/ASAG2024](https://huggingface.co/datasets/Meyerger/ASAG2024) — 45k train / 5.7k validation rows  
**Base model:** TheBloke/Mistral-7B-Instruct-v0.2-GPTQ (4-bit quantized)  
**Method:** QLoRA — only 0.79% of parameters are trained

## Step 0 — Mount Google Drive
Keeps checkpoints and the repo persistent across session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PROJECT_DIR = '/content/drive/MyDrive/AnswerReviewer'
CHECKPOINT_DIR    = f'{DRIVE_PROJECT_DIR}/model'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print(f'Drive mounted. Checkpoints will be saved to: {CHECKPOINT_DIR}')

## Step 1 — Sync Notebook from GitHub
Fetches only the latest `QLoRA_Fine-Tuning.ipynb` from your GitHub repo into Drive.
Every time you push a change locally, re-running this cell gets the update — no manual re-upload needed.

> **Before running:** add your GitHub PAT to Colab Secrets  
> `🔑 Secrets (left sidebar) → + Add new secret → Name: GITHUB_TOKEN → Value: your PAT → toggle ON`

In [ ]:
import os
from google.colab import userdata

# --- Config ---
GITHUB_USERNAME = "Aamir377300"
REPO_NAME       = "fine-tune"
NOTEBOOK_FILE   = "QLoRA_Fine-Tuning.ipynb"
# ---------------

GITHUB_TOKEN  = userdata.get('GITHUB_TOKEN')
NOTEBOOK_DIR  = f'/content/drive/MyDrive/AnswerReviewer'
NOTEBOOK_PATH = f'{NOTEBOOK_DIR}/{NOTEBOOK_FILE}'
RAW_URL       = f'https://{GITHUB_TOKEN}@raw.githubusercontent.com/{GITHUB_USERNAME}/{REPO_NAME}/master/{NOTEBOOK_FILE}'

os.makedirs(NOTEBOOK_DIR, exist_ok=True)

# Always fetch the latest version of the notebook from GitHub
!curl -s -o "{NOTEBOOK_PATH}" "{RAW_URL}"

print(f'Notebook synced from GitHub → {NOTEBOOK_PATH}')
print('Latest QLoRA_Fine-Tuning.ipynb is ready.')

## Step 2 — Install Dependencies

In [ ]:
!pip install -q transformers peft datasets accelerate bitsandbytes optimum auto-gptq huggingface_hub
print('All packages installed!')

## Step 3 — Keep Colab Alive
Paste this in your **browser console** (`F12` → Console tab) to prevent idle disconnection:

In [ ]:
# Paste this in your browser console, NOT here:
#
# function ClickConnect(){
#   console.log('Keeping Colab alive...');
#   document.querySelector('#top-toolbar > colab-connect-button')
#     .shadowRoot.querySelector('#connect').click();
# }
# setInterval(ClickConnect, 60000);
#
# Training takes ~2-3 hours on T4. Checkpoints save to Drive every epoch.
print('Read the comment above and paste the JS in your browser console.')

## Step 4 — Imports

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import prepare_model_for_kbit_training, LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset
import transformers
import warnings
warnings.filterwarnings('ignore')
from transformers import logging
logging.set_verbosity_error()
print('Imports done.')

## Step 5 — Load & Explore the ASAG2024 Dataset
Loads directly from HuggingFace — no CSV needed.

In [ ]:
# Load only the main parquet files directly — avoids the student_ranking schema conflict
from datasets import load_dataset, DatasetDict

dataset = DatasetDict({
    'train':      load_dataset('parquet', data_files='hf://datasets/Meyerger/ASAG2024/train.parquet',      split='train'),
    'validation': load_dataset('parquet', data_files='hf://datasets/Meyerger/ASAG2024/validation.parquet', split='train'),
    'test':       load_dataset('parquet', data_files='hf://datasets/Meyerger/ASAG2024/test.parquet',       split='train'),
})

print(dataset)
print('\nColumn names:', dataset['train'].column_names)
print('\nSample row:')
print(dataset['train'][0])

## Step 6 — Format Dataset into Instruction Prompts

Each row is converted into a Mistral instruction prompt:
```
[INST] You are an answer reviewer...
Question: ...
Reference Answer: ...
Student Answer: ...
[/INST]
Grade: correct/partially_correct/incorrect
Feedback: ...
```

The `normalized_grade` (0–1) is bucketed into 3 labels:
- `correct` → score >= 0.75
- `partially_correct` → 0.25 <= score < 0.75
- `incorrect` → score < 0.25

In [ ]:
SYSTEM_PROMPT = """You are an expert answer reviewer. You are given a question, a reference answer, and a student's answer.
Your task is to evaluate the student's answer by comparing it to the reference answer.
Respond with:
- Grade: one of [correct, partially_correct, incorrect]
- Feedback: a brief explanation of why the student's answer is correct, partially correct, or incorrect."""

def get_grade_label(normalized_grade):
    """Convert normalized score (0-1) to a text label."""
    if normalized_grade >= 0.75:
        return "correct"
    elif normalized_grade >= 0.25:
        return "partially_correct"
    else:
        return "incorrect"

def get_feedback(grade_label, question, reference_answer, student_answer):
    """Generate a simple feedback string based on the grade label."""
    if grade_label == "correct":
        return "The student's answer correctly captures the key concepts from the reference answer."
    elif grade_label == "partially_correct":
        return "The student's answer captures some aspects of the reference answer but is missing key details or contains inaccuracies."
    else:
        return "The student's answer does not align with the reference answer and misses the key concepts."

def format_example(row):
    """Format a dataset row into a Mistral instruction prompt."""
    question         = row['question'].strip()
    reference_answer = row['reference_answer'].strip()
    student_answer   = row['provided_answer'].strip()
    normalized_grade = row['normalized_grade']

    grade_label = get_grade_label(normalized_grade)
    feedback    = get_feedback(grade_label, question, reference_answer, student_answer)

    prompt = f"""<s>[INST] {SYSTEM_PROMPT}

Question: {question}
Reference Answer: {reference_answer}
Student Answer: {student_answer}
[/INST]
Grade: {grade_label}
Feedback: {feedback}</s>"""

    return {"text": prompt}

# Apply formatting to train and validation splits
# Using a subset for faster training — increase if you have Colab Pro / more time
TRAIN_SAMPLES = 5000   # out of 45k — enough to learn the task well
EVAL_SAMPLES  = 500    # out of 5.7k

train_data = dataset['train'].shuffle(seed=42).select(range(TRAIN_SAMPLES))
eval_data  = dataset['validation'].shuffle(seed=42).select(range(EVAL_SAMPLES))

train_formatted = train_data.map(format_example)
eval_formatted  = eval_data.map(format_example)

print(f'Train examples: {len(train_formatted)}')
print(f'Eval examples:  {len(eval_formatted)}')
print('\nSample formatted prompt:')
print(train_formatted[0]['text'])

## Step 7 — Load Base Model
Downloads ~4GB from HuggingFace. Takes 3–5 minutes on first run.

In [ ]:
model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    trust_remote_code=False,
    revision="main"
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

print('Model and tokenizer loaded!')

## Step 8 — Test Base Model (Before Fine-tuning)
See what the untuned model outputs — useful for comparison after training.

In [ ]:
test_prompt = f"""[INST] {SYSTEM_PROMPT}

Question: What is photosynthesis?
Reference Answer: Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce oxygen and energy in the form of glucose.
Student Answer: Photosynthesis is when plants make food from sunlight.
[/INST]"""

model.eval()
inputs = tokenizer(test_prompt, return_tensors="pt")
outputs = model.generate(
    input_ids=inputs["input_ids"].to("cuda"),
    max_new_tokens=150,
    temperature=0.1
)
response = tokenizer.batch_decode(outputs)[0].split('[/INST]')[-1].strip().replace('</s>', '').strip()
print('--- Base model output (before fine-tuning) ---')
print(response)

## Step 9 — Prepare Model for QLoRA Training

In [ ]:
model.train()
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

# LoRA config
config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["q_proj", "v_proj"],  # targeting both q and v for better grading accuracy
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

## Step 10 — Tokenize Dataset

In [ ]:
def tokenize_function(examples):
    tokenizer.truncation_side = "left"
    return tokenizer(
        examples["text"],
        return_tensors="np",
        truncation=True,
        max_length=512,
        padding=False
    )

tokenized_train = train_formatted.map(tokenize_function, batched=True, remove_columns=train_formatted.column_names)
tokenized_eval  = eval_formatted.map(tokenize_function,  batched=True, remove_columns=eval_formatted.column_names)

data_collator = transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)

print(f'Tokenized train: {len(tokenized_train)} examples')
print(f'Tokenized eval:  {len(tokenized_eval)} examples')

## Step 11 — Fine-tune the Model
With 5000 training examples on a T4 GPU, this takes approximately **2–3 hours**.
Checkpoints are saved to Google Drive after every epoch.

In [ ]:
lr         = 2e-4
batch_size = 2    # lower than before because prompts are longer
num_epochs = 3    # 3 epochs is enough for this task size

training_args = transformers.TrainingArguments(
    output_dir=CHECKPOINT_DIR,
    learning_rate=lr,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=num_epochs,
    weight_decay=0.01,
    logging_strategy="epoch",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    gradient_accumulation_steps=8,   # effective batch size = 2 x 8 = 16
    warmup_steps=10,
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",                # disable wandb
)

trainer = transformers.Trainer(
    model=model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    args=training_args,
    data_collator=data_collator
)

model.config.use_cache = False
trainer.train()
model.config.use_cache = True

print('Training complete!')

## Step 12 — Test the Fine-tuned Model

In [ ]:
def review_answer(question, reference_answer, student_answer, max_new_tokens=200):
    """Run the fine-tuned model on a question/answer pair and return the grade + feedback."""
    prompt = f"""[INST] {SYSTEM_PROMPT}

Question: {question}
Reference Answer: {reference_answer}
Student Answer: {student_answer}
[/INST]"""

    model.eval()
    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(
        input_ids=inputs["input_ids"].to("cuda"),
        max_new_tokens=max_new_tokens,
        temperature=0.1,
        do_sample=True
    )
    response = tokenizer.batch_decode(outputs)[0].split('[/INST]')[-1].strip().replace('</s>', '').strip()
    return response

# --- Test 1: Correct answer ---
print('=== Test 1: Correct Answer ===')
print(review_answer(
    question="What is photosynthesis?",
    reference_answer="Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce oxygen and energy in the form of glucose.",
    student_answer="Photosynthesis is the process plants use to convert sunlight, water, and CO2 into glucose and oxygen."
))

print()

# --- Test 2: Partially correct answer ---
print('=== Test 2: Partially Correct Answer ===')
print(review_answer(
    question="What is photosynthesis?",
    reference_answer="Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce oxygen and energy in the form of glucose.",
    student_answer="Photosynthesis is when plants make food from sunlight."
))

print()

# --- Test 3: Incorrect answer ---
print('=== Test 3: Incorrect Answer ===')
print(review_answer(
    question="What is photosynthesis?",
    reference_answer="Photosynthesis is the process by which plants use sunlight, water, and carbon dioxide to produce oxygen and energy in the form of glucose.",
    student_answer="Photosynthesis is how animals digest food."
))

## Step 13 — Push Fine-tuned Model to HuggingFace Hub
Uploads only the LoRA adapter weights (a few MB), not the full 7B model.

In [ ]:
from huggingface_hub import login

# Get your token from: https://huggingface.co/settings/tokens (needs Write permission)
HF_TOKEN    = "hf_xxxxxxxxxxxxxxxxxxxx"  # <-- paste your HF write token here
HF_USERNAME = "your-hf-username"         # <-- replace with your HuggingFace username

login(HF_TOKEN)
print('Logged in to HuggingFace!')

In [ ]:
model_id = f"{HF_USERNAME}/answer-reviewer"

model.push_to_hub(model_id)
trainer.push_to_hub(model_id)

print(f'Model pushed to: https://huggingface.co/{model_id}')

## (Optional) Resume Training from Checkpoint
If Colab disconnected mid-training, use this to continue from the last saved checkpoint.

In [ ]:
# Uncomment and run this block to resume training from the last checkpoint

# trainer.train(resume_from_checkpoint=True)
# print('Resumed training from last checkpoint!')

## (Optional) Load Model from HuggingFace Hub
Use this in a new session to reload your fine-tuned model without retraining.

In [ ]:
# Uncomment to load your pushed model from HuggingFace Hub

# HF_USERNAME     = "your-hf-username"
# base_model_name = "TheBloke/Mistral-7B-Instruct-v0.2-GPTQ"
# adapter_model   = f"{HF_USERNAME}/answer-reviewer"
#
# model     = AutoModelForCausalLM.from_pretrained(base_model_name, device_map="auto", revision="main")
# model     = PeftModel.from_pretrained(model, adapter_model)
# tokenizer = AutoTokenizer.from_pretrained(base_model_name, use_fast=True)
# print('Fine-tuned model loaded from Hub!')